# Практическая работа №4. Вычислительные эксперименты в PySpark

**Выполнил:** Комисарик М.А., группа ИКБО-20-23

**Цель работы** — исследовать физический механизм выполнения задач в Apache Spark: связь преобразований и действий с Jobs, Stages и Tasks, влияние числа партиций на параллелизм, перемешивание данных при широких преобразованиях, выбор стратегии JOIN, повторное вычисление и эффект `cache()`/`persist()`.

Значения времени зависят от стенда и не являются эталонными; варианты сравниваются внутри одной сессии, а наблюдаемое время связывается с физическим планом и Spark UI.

In [1]:
import datetime
print("Дата выполнения:", datetime.date.today().strftime("%d.%m.%Y"))

Дата выполнения: 08.11.2026


## 3. Экспериментальный стенд и подготовка данных

### 3.2. Staging-каталог

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("ecommerce.sqlite")
STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

TABLES = [
    "geolocation", "order_items", "products",
    "sellers", "product_category_name_translation"
]

with sqlite3.connect(DB_PATH) as con:
    for table in TABLES:
        out = STAGING / f"{table}.csv"
        first = True
        total = 0
        for chunk in pd.read_sql_query(
            f"SELECT * FROM {table}", con, chunksize=100_000
        ):
            chunk.to_csv(
                out, index=False,
                mode="w" if first else "a",
                header=first, encoding="utf-8"
            )
            total += len(chunk)
            first = False
        print(f"{table:40s} {total:>9d} rows")

geolocation                                1000163 rows
order_items                                 112650 rows
products                                     32951 rows
sellers                                       3095 rows
product_category_name_translation               71 rows


### 3.3. SparkSession с контролируемой конфигурацией

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Practice4_SparkExecution")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Spark UI:", spark.sparkContext.uiWebUrl)

Master: local[*]
Default parallelism: 20
Shuffle partitions: 8
AQE: false
Spark UI: http://KREZON-PC:4040


In [4]:
# Сведения для паспорта стенда (таблица 2)
import os
import sys
import platform

print("ОС:", platform.platform())
print("Python:", sys.version.split()[0])
print("Java:", spark.sparkContext._jvm.System.getProperty("java.version"))
print("Spark:", spark.version)
print("Логических CPU:", os.cpu_count())

ОС: Windows-10-10.0.26200-SP0
Python: 3.11.16
Java: 21.0.9
Spark: 4.2.0
Логических CPU: 20


**Таблица 2. Паспорт экспериментального стенда**

| Параметр | Фактическое значение |
|---|---|
| ОС | Windows 11 (10.0.26200) |
| Python | 3.11.16 |
| Java | 21.0.9 |
| PySpark / Spark | 4.2.0 |
| Логических CPU | 20 |
| Оперативная память | 31.7 ГБ |
| spark.master | local[*] |
| spark.default.parallelism | 20 |
| spark.sql.shuffle.partitions | 8 |
| spark.sql.adaptive.enabled | false |

### 3.4. Перевод исходных таблиц в Parquet

In [5]:
PARQUET = Path("spark_input_parquet")
PARQUET.mkdir(exist_ok=True)

for name in TABLES:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(STAGING / f"{name}.csv"))
    )
    df.write.mode("overwrite").parquet(str(PARQUET / name))

geo = spark.read.parquet(str(PARQUET / "geolocation"))
items = spark.read.parquet(str(PARQUET / "order_items"))
products = spark.read.parquet(str(PARQUET / "products"))
sellers = spark.read.parquet(str(PARQUET / "sellers"))
translations = spark.read.parquet(
    str(PARQUET / "product_category_name_translation")
)

print(geo.count(), items.count(), products.count(), sellers.count())

1000163 112650 32951 3095


## 4. Spark UI как измерительный инструмент

### 4.1. Маркировка экспериментов

In [6]:
sc = spark.sparkContext
sc.setJobGroup(
    "P4_BASELINE",
    "Практика 4: базовая проверка Spark UI",
    interruptOnCancel=True
)
_ = items.count()

**Наблюдение в Spark UI.** Job группы `P4_BASELINE` состоит из 2 стадий; число Tasks по стадиям: 4 → 1. Число задач первой стадии совпадает с числом входных партиций `order_items` (4), финальная стадия из одной задачи складывает частичные подсчёты.

## 5. Исследование ленивых вычислений

### 5.1. Цепочка transformations без Action

In [7]:
sc.setJobGroup("P4_LAZY", "Lazy evaluation")
geo_prepared = (
    geo
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_state",
        "geolocation_lat",
        "geolocation_lng"
    )
    .filter(F.col("geolocation_state").isNotNull())
    .withColumn("abs_lat", F.abs(F.col("geolocation_lat")))
)
print(type(geo_prepared).__name__)

DataFrame


**Наблюдение в Spark UI.** После выполнения ячейки на вкладке Jobs нового Job с группой `P4_LAZY` нет.

### 5.2. Действие как момент запуска вычисления

In [8]:
from time import perf_counter

t0 = perf_counter()
n = geo_prepared.count()
elapsed = perf_counter() - t0
print("Rows:", n)
print(f"Elapsed: {elapsed:.3f} s")

Rows: 1000163
Elapsed: 0.556 s


**Вывод.** Три преобразования не запустили ни одного Job: Spark только построил план. Job группы `P4_LAZY` появился в Spark UI лишь после действия `count()`, которое заняло 0,556 с. Число строк осталось 1 000 163, так как в `geolocation_state` нет пропусков.

## 6. Исследование партиций и параллелизма

### 6.1. Число партиций и распределение строк

In [9]:
print("Input partitions:", geo.rdd.getNumPartitions())
print("Default parallelism:", sc.defaultParallelism)

partition_sizes = (
    geo
    .select(F.spark_partition_id().alias("pid"))
    .groupBy("pid")
    .count()
    .orderBy("pid")
)
partition_sizes.show(100, truncate=False)

Input partitions: 16
Default parallelism: 20
+---+-----+
|pid|count|
+---+-----+
|0  |63727|
|1  |62834|
|2  |64015|
|3  |63035|
|4  |63121|
|5  |62363|
|6  |62677|
|7  |62580|
|8  |64612|
|9  |61012|
|10 |62301|
|11 |54211|
|12 |61517|
|13 |65521|
|14 |66154|
|15 |60483|
+---+-----+



**Вывод.** `geolocation` прочитана в 16 партиций при `defaultParallelism` = 20. В Spark UI первая стадия этого запроса содержит 16 Tasks: одна задача обрабатывает одну партицию. Поскольку parallelism ≤ min(C, p) = min(20, 16), при чтении одновременно загружены только 16 из 20 вычислительных слотов.

### 6.2. Перекос по geolocation_state

In [10]:
state_counts = (
    geo
    .groupBy("geolocation_state")
    .count()
    .orderBy(F.desc("count"))
)
state_counts.show(5, truncate=False)

+-----------------+------+
|geolocation_state|count |
+-----------------+------+
|SP               |404268|
|MG               |126336|
|RJ               |121169|
|RS               |61851 |
|PR               |57859 |
+-----------------+------+
only showing top 5 rows


**Вывод.** Штаты распределены крайне неравномерно: SP — 404 268 строк (40,42%), следующие MG и RJ — около 12% каждый. При разбиении по этому ключу партиция с SP неизбежно окажется самой крупной: перекос является свойством данных, а не Spark.

## 7. Сравнение repartition() и coalesce()

### 7.1. Контролируемое уменьшение партиций

In [11]:
skewed16 = geo.repartition(16, "geolocation_state")
repart4 = skewed16.repartition(4)
coal4 = skewed16.coalesce(4)

print("skewed16:", skewed16.rdd.getNumPartitions())
print("repart4:", repart4.rdd.getNumPartitions())
print("coal4:", coal4.rdd.getNumPartitions())

skewed16: 16
repart4: 4
coal4: 4


### 7.2. Сравнение физического плана

In [12]:
print("=== repartition(4) ===")
repart4.explain("formatted")
print("=== coalesce(4) ===")
coal4.explain("formatted")

=== repartition(4) ===
== Physical Plan ==
Exchange (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [geolocation_id#141, geolocation_zip_code_prefix#142, geolocation_lat#143, geolocation_lng#144, geolocation_city#145, geolocation_state#146]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/geolocation]
ReadSchema: struct<geolocation_id:int,geolocation_zip_code_prefix:int,geolocation_lat:double,geolocation_lng:double,geolocation_city:string,geolocation_state:string>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [geolocation_id#141, geolocation_zip_code_prefix#142, geolocation_lat#143, geolocation_lng#144, geolocation_city#145, geolocation_state#146]

(3) Exchange
Input [6]: [geolocation_id#141, geolocation_zip_code_prefix#142, geolocation_lat#143, geolocation_lng#144, geolocation_city#145, geolocation_state#146]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=

### 7.3. Размеры выходных партиций

In [13]:
def partition_profile(df):
    return (
        df
        .select(F.spark_partition_id().alias("pid"))
        .groupBy("pid")
        .count()
        .orderBy("pid")
    )


print("repartition(4)")
partition_profile(repart4).show()
print("coalesce(4)")
partition_profile(coal4).show()

repartition(4)
+---+------+
|pid| count|
+---+------+
|  0|250040|
|  1|250041|
|  2|250041|
|  3|250041|
+---+------+

coalesce(4)
+---+------+
|pid| count|
+---+------+
|  0|195064|
|  1|626485|
|  2| 90152|
|  3| 88462|
+---+------+



**Таблица 4. Распределение строк по партициям**

| Вариант | min rows | max rows | mean rows | Комментарий |
|---|---|---|---|---|
| repartition(4) | 250 040 | 250 041 | 250 041 | Полное перераспределение (`Exchange RoundRobinPartitioning(4)`), партиции практически равны |
| coalesce(4) | 88 462 | 626 485 | 250 041 | Оператор `Coalesce` склеивает соседние партиции без нового обмена, перекос по штатам сохраняется |

**Вывод.** Оба варианта дают 4 партиции и 1 000 163 строки, но путь к результату разный. `repartition(4)` добавляет в план второй `Exchange` и выравнивает партиции ценой полного перемешивания. `coalesce(4)` добавляет только `Coalesce`: соседние партиции объединяются без обмена, поэтому самая крупная партиция, содержащая SP, в 7 раз больше самой маленькой.

## 8. Перемешивание данных на примере groupBy()

### 8.1. Агрегация order_items по продавцам

In [14]:
sc.setJobGroup("P4_GROUPBY", "Shuffle: groupBy seller_id")
seller_stats = (
    items
    .groupBy("seller_id")
    .agg(
        F.count("*").alias("items"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
    .orderBy(F.desc("revenue"))
)
seller_stats.show(5, truncate=False)

+--------------------------------+-----+---------+---------+
|seller_id                       |items|revenue  |avg_price|
+--------------------------------+-----+---------+---------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156 |229472.63|198.51   |
|53243585a1d6dc2643021fd1853d8905|410  |222776.05|543.36   |
|4a3ca9315b744ce9f8e9374361493884|1987 |200472.92|100.89   |
|fa1c13f2614d7b5c4749cbc52fecda94|586  |194042.03|331.13   |
|7c67e1448b00f6e969d365cea6b010ab|1364 |187923.89|137.77   |
+--------------------------------+-----+---------+---------+
only showing top 5 rows


**Наблюдение в Spark UI.** Число стадий: 2; Tasks по стадиям: 4 → 8; Shuffle Write предыдущей стадии: 438,6 КиБ; Shuffle Read следующей стадии: 438,6 КиБ.

### 8.2. Физический план groupBy()

In [15]:
seller_stats.explain("formatted")

== Physical Plan ==
* Sort (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- Exchange (4)
         +- * HashAggregate (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [seller_id#150, price#152]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/order_items]
ReadSchema: struct<seller_id:string,price:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [seller_id#150, price#152]

(3) HashAggregate [codegen id : 1]
Input [2]: [seller_id#150, price#152]
Keys [1]: [seller_id#150]
Functions [3]: [partial_count(1), partial_sum(price#152), partial_avg(price#152)]
Aggregate Attributes [4]: [count#307L, sum#308, sum#309, count#310L]
Results [5]: [seller_id#150, count#311L, sum#312, sum#313, count#314L]

(4) Exchange
Input [5]: [seller_id#150, count#311L, sum#312, sum#313, count#314L]
Arguments: hashpartitioning(seller_id#150, 8), ENSURE_REQUIREMENTS, [plan_id=

**Вывод.** Группировка выполняется в две фазы: `HashAggregate` с функциями `partial_count`, `partial_sum`, `partial_avg` агрегирует строки внутри каждой входной партиции, затем `Exchange hashpartitioning(seller_id, 8)` перераспределяет частичные результаты по ключу, и финальный `HashAggregate` собирает итог. В Spark UI эта граница видна как переход между стадиями с Shuffle Write на одной и Shuffle Read на следующей; стадия после обмена содержит 8 задач по числу `spark.sql.shuffle.partitions`.

## 9. Анализ плана через explain("formatted")

In [16]:
plan_demo = (
    items
    .filter(F.col("price") >= 100)
    .select("seller_id", "price", "freight_value")
    .withColumn("gross", F.col("price") + F.col("freight_value"))
    .groupBy("seller_id")
    .agg(F.round(F.sum("gross"), 2).alias("gross_sum"))
)
plan_demo.explain("formatted")

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [seller_id#150, price#152, freight_value#153]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(price), GreaterThanOrEqual(price,100.0)]
ReadSchema: struct<seller_id:string,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [seller_id#150, price#152, freight_value#153]

(3) Filter [codegen id : 1]
Input [3]: [seller_id#150, price#152, freight_value#153]
Condition : (isnotnull(price#152) AND (price#152 >= 100.0))

(4) Project [codegen id : 1]
Output [2]: [seller_id#150, (price#152 + freight_value#153) AS gross#336]
Input [3]: [seller_id#150, price#152, freight_value#153]

(5) HashAggregate [codegen id : 1]
Input [2]:

**Вывод.** План читается снизу вверх: `Scan parquet` читает только столбцы `seller_id`, `price`, `freight_value`, а условие `price >= 100` передаётся в источник (`PushedFilters`); `Filter` отбирает строки; `Project` вычисляет `gross`; частичный `HashAggregate` уменьшает объём данных ещё до обмена; `Exchange hashpartitioning(seller_id, 8)` перераспределяет данные по ключу; финальный `HashAggregate` завершает агрегацию. Фильтрация и вычисление выполняются до перемешивания, поэтому передаётся меньше данных.

## 10. Sort Merge Join и Broadcast Hash Join

### 10.1. Sort Merge Join при отключённом broadcast

In [17]:
sc.setJobGroup("P4_SMJ", "Sort Merge Join: items x products")
smj = (
    items.hint("merge")
    .join(products.hint("merge"), on="product_id", how="inner")
)
smj.explain("formatted")

t0 = perf_counter()
print("Rows:", smj.count())
print(f"Elapsed: {perf_counter() - t0:.3f} s")

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]

(3) Filter [cod

### 10.2. Broadcast Hash Join

In [18]:
sc.setJobGroup("P4_BHJ", "Broadcast Hash Join: items x products")
bhj = items.join(
    F.broadcast(products),
    on="product_id",
    how="inner"
)
bhj.explain("formatted")

t0 = perf_counter()
print("Rows:", bhj.count())
print(f"Elapsed: {perf_counter() - t0:.3f} s")

== Physical Plan ==
* Project (9)
+- * BroadcastHashJoin Inner BuildRight (8)
   :- * Filter (3)
   :  +- * ColumnarToRow (2)
   :     +- Scan parquet  (1)
   +- BroadcastExchange (7)
      +- * Filter (6)
         +- * ColumnarToRow (5)
            +- Scan parquet  (4)


(1) Scan parquet 
Output [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 2]
Input [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]

(3) Filter [codegen id : 2]
Input [7]: [order_id#147, order_item_id#148, product

**Таблица 6. Сравнение JOIN-экспериментов**

| Показатель | Sort Merge Join | Broadcast Hash Join |
|---|---|---|
| Строк результата | 112 650 | 112 650 |
| Exchange большой стороны | есть: `hashpartitioning(product_id, 8)` | нет |
| Shuffle Read / Write | 3,74 МиБ / 3,74 МиБ | 236 Б / 236 Б |
| Jobs / Stages | 1 / 4 | 2 / 3 |
| Время, с | 0,711 | 0,448 |

**Вывод.** Одинаковое число строк (112 650) подтверждает логическую эквивалентность запросов. Sort Merge Join перераспределил обе стороны по `product_id` и отсортировал их (`Exchange` и `Sort` с каждой стороны): в Spark UI это две стадии чтения с Shuffle Write (2,68 МиБ для `order_items` и 1,06 МиБ для `products`) перед стадией соединения из 8 задач. Broadcast Hash Join передал `products` всем задачам через `BroadcastExchange` отдельным Job, а большая сторона не перемешивалась: весь Shuffle составил 236 байт, приходящихся на финальный подсчёт строк. На данном стенде время составило 0,711 с против 0,448 с; вывод условен и относится к справочнику размером около 33 тыс. строк.

## 11. Исследование cache() / persist()

### 11.1. Повторно используемый промежуточный DataFrame

In [19]:
spark.catalog.clearCache()

products_small = products.select(
    "product_id", "product_category_name"
)
sellers_small = sellers.select(
    "seller_id", "seller_state"
)

joined = (
    items
    .join(F.broadcast(products_small), "product_id", "left")
    .join(F.broadcast(sellers_small), "seller_id", "left")
    .withColumn(
        "gross_value",
        F.col("price") + F.col("freight_value")
    )
)


def action_category(df):
    return (
        df.groupBy("product_category_name")
        .agg(F.sum("gross_value").alias("gross"))
        .agg(F.sum("gross").alias("checksum"))
        .first()[0]
    )


def action_state(df):
    return (
        df.groupBy("seller_state")
        .agg(F.sum("gross_value").alias("gross"))
        .agg(F.sum("gross").alias("checksum"))
        .first()[0]
    )

### 11.2. Измерение без persist()

In [20]:
sc.setJobGroup("P4_NOCACHE", "Два действия без сохранения")
t0 = perf_counter()
_ = action_category(joined)
t1 = perf_counter()
_ = action_state(joined)
t2 = perf_counter()
print(f"Action 1: {t1 - t0:.3f} s")
print(f"Action 2: {t2 - t1:.3f} s")
print("is_cached:", joined.is_cached)

Action 1: 0.629 s
Action 2: 0.403 s
is_cached: False


### 11.3. persist(MEMORY_AND_DISK) и материализация

In [21]:
sc.setJobGroup("P4_CACHE", "persist + повторные действия")
cached = joined.persist(StorageLevel.MEMORY_AND_DISK)
print("После persist:", cached.is_cached)

# Materialization: первый Action заполняет кэш
t0 = perf_counter()
materialized_rows = cached.count()
t1 = perf_counter()

# Повторные Actions используют материализованный DataFrame
_ = action_category(cached)
t2 = perf_counter()
_ = action_state(cached)
t3 = perf_counter()

print("Rows:", materialized_rows)
print(f"Materialization: {t1 - t0:.3f} s")
print(f"Cached action 1: {t2 - t1:.3f} s")
print(f"Cached action 2: {t3 - t2:.3f} s")

cached.unpersist(blocking=True)
print("После unpersist:", cached.is_cached)

После persist: True
Rows: 112650
Materialization: 0.727 s
Cached action 1: 0.384 s
Cached action 2: 0.196 s
После unpersist: False


**Таблица 7. Сравнение повторных вычислений с кэшем и без него**

| Вариант | Подготовка/материализация, с | Action 1, с | Action 2, с |
|---|---|---|---|
| Без persist | — | 0,629 | 0,403 |
| MEMORY_AND_DISK | 0,727 | 0,384 | 0,196 |

**Вывод.** Вызов `persist()` сам ничего не вычислил: кэш заполнился при первом действии `count()` за 0,727 с. Повторные действия над сохранённым DataFrame заняли 0,384 и 0,196 с против 0,629 и 0,403 с без кэша, так как им не нужно заново читать источники и выполнять JOIN. В Spark UI это видно по структуре Jobs: без кэша каждое действие заново запускало Jobs рассылки `products` и `sellers`, а с кэшем они выполнились один раз при материализации, и действия читали сохранённые данные (Input 8,4 МиБ). С учётом материализации суммарное время составило 1,307 с против 1,032 с без кэша, то есть при двух повторных действиях кэш не окупился. После `unpersist()` объект исчез со вкладки Storage.

## 12. Влияние числа партиций на время выполнения

### 12.1. Контролируемый бенчмарк

In [22]:
from statistics import median

N = 2_000_000
P_VALUES = [1, 2, 4, 8, 16, 32]
spark.conf.set("spark.sql.shuffle.partitions", "8")


def run_benchmark(p, repeats=3):
    times = []
    checksum = None
    for _ in range(repeats):
        df = (
            spark.range(0, N, 1, numPartitions=p)
            .withColumn("key", (F.col("id") % 1000).cast("int"))
            .withColumn("value", F.sqrt(F.col("id") + 1.0))
        )
        t0 = perf_counter()
        checksum = (
            df.groupBy("key")
            .agg(F.sum("value").alias("s"))
            .agg(F.sum("s").alias("checksum"))
            .first()[0]
        )
        times.append(perf_counter() - t0)
    return times, median(times), checksum


# Прогрев
_ = run_benchmark(4, repeats=1)

results = []
for p in P_VALUES:
    times, med, checksum = run_benchmark(p, repeats=3)
    results.append((p, *times, med))
    print(p, times, med, checksum)

1 [0.21865900000557303, 0.07884439988993108, 0.10934780002571642] 0.10934780002571642 1885618790.063052
2 [0.25786749995313585, 0.19889579992741346, 0.1531843999400735] 0.19889579992741346 1885618790.0630507
4 [0.19193830015137792, 0.19390109996311367, 0.1355343998875469] 0.19193830015137792 1885618790.0630512
8 [0.25396010000258684, 0.1968785000499338, 0.2405336000956595] 0.2405336000956595 1885618790.0630512
16 [0.2838930001016706, 0.22343280003406107, 0.23348390008322895] 0.23348390008322895 1885618790.0630512
32 [0.3904367999639362, 0.3338512999471277, 0.3057895000092685] 0.3338512999471277 1885618790.0630512


### 12.2. Относительное ускорение

In [23]:
t_1 = results[0][-1]
for p, *runs, med in results:
    print(f"p={p:2d}  runs={[round(t, 3) for t in runs]}  median={med:.3f} s  S(p)={t_1 / med:.2f}")

p= 1  runs=[0.219, 0.079, 0.109]  median=0.109 s  S(p)=1.00
p= 2  runs=[0.258, 0.199, 0.153]  median=0.199 s  S(p)=0.55
p= 4  runs=[0.192, 0.194, 0.136]  median=0.192 s  S(p)=0.57
p= 8  runs=[0.254, 0.197, 0.241]  median=0.241 s  S(p)=0.45
p=16  runs=[0.284, 0.223, 0.233]  median=0.233 s  S(p)=0.47
p=32  runs=[0.39, 0.334, 0.306]  median=0.334 s  S(p)=0.33


**Таблица 8. Влияние числа входных партиций на время**

| p | Run 1, с | Run 2, с | Run 3, с | Median, с | S(p)=T(1)/T(p) |
|---|---|---|---|---|---|
| 1 | 0,219 | 0,079 | 0,109 | 0,109 | 1,00 |
| 2 | 0,258 | 0,199 | 0,153 | 0,199 | 0,55 |
| 4 | 0,192 | 0,194 | 0,136 | 0,192 | 0,57 |
| 8 | 0,254 | 0,197 | 0,241 | 0,241 | 0,45 |
| 16 | 0,284 | 0,223 | 0,233 | 0,233 | 0,47 |
| 32 | 0,390 | 0,334 | 0,306 | 0,334 | 0,33 |

**Вывод.** При `defaultParallelism` = 20 минимальное медианное время 0,109 с получено при p = 1, а при всех p > 1 ускорение S(p) меньше единицы. Причина видна в Spark UI: при p = 1 все строки уже находятся в одной партиции, и Spark не добавляет `Exchange`, поэтому Job состоит из одной стадии без перемешивания. Начиная с p = 2 появляется обмен, и Shuffle Write растёт пропорционально p — от 24,4 КиБ при p = 2 до 392 КиБ при p = 32, так как каждая задача записывает частичные суммы по всем 1000 ключам. При N = 2 млн вычисления занимают доли секунды, поэтому выигрыш от параллелизма меньше затрат на перемешивание и планирование задач. Оптимальное p зависит от стенда, операции и объёма данных; для оценки масштабирования потребовалось бы увеличить N (методические указания допускают 5 млн).

### 12.3. Число партиций после перемешивания

In [24]:
geo8 = geo.repartition(8)
Q_VALUES = [2, 4, 8, 16, 32]
for q in Q_VALUES:
    spark.conf.set("spark.sql.shuffle.partitions", str(q))
    t0 = perf_counter()
    rows = (
        geo8.groupBy("geolocation_state")
        .count()
        .count()
    )
    elapsed = perf_counter() - t0
    print(f"q={q:2d}; groups={rows}; time={elapsed:.3f}s")

# Возвращаем базовую настройку
spark.conf.set("spark.sql.shuffle.partitions", "8")

q= 2; groups=27; time=0.556s
q= 4; groups=27; time=0.369s
q= 8; groups=27; time=0.328s
q=16; groups=27; time=0.392s
q=32; groups=27; time=0.484s


**Вывод.** При всех значениях `spark.sql.shuffle.partitions` получено 27 групп; менялось только число партиций после обмена, а входные партиции `geo8` оставались равными 8. Параметр задаёт разбиение после перемешивания, а не разбиение исходных файлов. Наименьшее время (0,328 с) получено при q = 8; при q = 32 оно выросло до 0,484 с, так как часть из 32 партиций гарантированно пуста (групп всего 27), и задачи по ним создают лишние накладные расходы.

## 13. Партиции и маленькие файлы

### 13.1. Запись с разным числом партиций

In [25]:
OUT_SMALL = Path("small_files_experiment")
OUT_SMALL.mkdir(exist_ok=True)


def write_and_measure(p):
    path = OUT_SMALL / f"p_{p}"
    (
        items.repartition(p)
        .write.mode("overwrite")
        .parquet(str(path))
    )
    files = list(path.glob("part-*.parquet"))
    sizes = [f.stat().st_size for f in files]
    total_mib = sum(sizes) / 1024**2
    avg_kib = (sum(sizes) / len(sizes) / 1024) if sizes else 0
    return len(files), total_mib, avg_kib


for p in [1, 4, 16, 64]:
    print(p, write_and_measure(p))

1 (1, 6.456236839294434, 6611.1865234375)
4 (4, 7.432483673095703, 1902.7158203125)
16 (16, 8.732264518737793, 558.8649291992188)
64 (64, 10.285961151123047, 164.57537841796875)


**Таблица 9. Связь партиций и part-файлов**

| p | part-файлов | Общий размер, MiB | Средний файл, KiB |
|---|---|---|---|
| 1 | 1 | 6,46 | 6611,2 |
| 4 | 4 | 7,43 | 1902,7 |
| 16 | 16 | 8,73 | 558,9 |
| 64 | 64 | 10,29 | 164,6 |

**Вывод.** Число part-файлов равно числу партиций перед записью: каждая задача записи создаёт свой файл. Средний размер файла уменьшается почти пропорционально p (с 6611,2 до 164,6 КиБ), а общий объём вырос с 6,46 до 10,29 МиБ: каждый файл хранит собственные метаданные и словари, а небольшие блоки хуже сжимаются. При большом p набор превращается во множество мелких файлов, которые дороже хранить, открывать и планировать при последующем чтении.

### 13.2. coalesce(1) не является универсальным решением

In [26]:
single = items.coalesce(1)
print("Partitions before write:", single.rdd.getNumPartitions())
# Выполнять только как учебный пример на небольшом order_items
(single.write.mode("overwrite")
 .parquet(str(OUT_SMALL / "coalesce_1")))

Partitions before write: 1


**Вывод.** `coalesce(1)` даёт один файл, но вся запись выполняется одной задачей. Для небольшой витрины это допустимо, для больших данных — узкое место, уничтожающее параллелизм записи.

## 14. Дополнительный эксперимент: Adaptive Query Execution

In [27]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "32")

adaptive = (
    geo
    .groupBy("geolocation_state")
    .count()
)
adaptive.explain("formatted")
adaptive.collect()
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [geolocation_state#146]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/geolocation]
ReadSchema: struct<geolocation_state:string>

(2) HashAggregate
Input [1]: [geolocation_state#146]
Keys [1]: [geolocation_state#146]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#1743L]
Results [2]: [geolocation_state#146, count#1744L]

(3) Exchange
Input [2]: [geolocation_state#146, count#1744L]
Arguments: hashpartitioning(geolocation_state#146, 32), ENSURE_REQUIREMENTS, [plan_id=3064]

(4) HashAggregate
Input [2]: [geolocation_state#146, count#1744L]
Keys [1]: [geolocation_state#146]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#1742L]
Results [2]: [geolocation_state#146, count(1)#1742L AS count#1735L]

(5) AdaptiveSparkP

In [28]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

**Вывод.** С включённым AQE корнем плана стал `AdaptiveSparkPlan`. В Spark UI стадия после перемешивания выполнилась одной задачей вместо заданных 32: AQE по runtime-статистике объединил мелкие партиции 27 групп. Это удобно на практике, но усложняет причинный анализ, поэтому основные опыты выполнены с AQE = false.

## 15. Журнал наблюдений по Spark UI

**Таблица 10. Журнал наблюдений по Spark UI**

| Эксперимент | Jobs | Stages | Tasks в ключевой Stage | Shuffle Write | Shuffle Read | Что подтверждает UI |
|---|---|---|---|---|---|---|
| groupBy seller_id | 1 | 2 | 8 (стадии: 4 → 8) | 438,6 КиБ | 438,6 КиБ | Две стадии с обменом по ключу между частичной и финальной агрегацией |
| repartition(4) | 1 | 4 | 4 (стадии: 16 → 16 → 4 → 8) | 173,8 КиБ | 173,8 КиБ | Два обмена: по штату на 16 партиций и round-robin на 4; тот же запрос с `coalesce(4)` содержит 3 стадии (16 → 4 → 8) без нового обмена |
| Sort Merge Join | 1 | 4 | 8 (стадии: 4 и 1 → 8 → 1) | 3,74 МиБ | 3,74 МиБ | Обе стороны перемешаны по `product_id` перед стадией соединения |
| Broadcast Hash Join | 2 | 3 | 4 (стадии: 1 \| 4 → 1) | 236 Б | 236 Б | Отдельный Job рассылки `products`; большая сторона не перемешивается |
| persist + повторный Action | 5 | 10 | 4 (чтение кэша, Input 8,4 МиБ) | 15,8 КиБ | 15,8 КиБ | Рассылка справочников выполнена один раз; повторные действия читают кэш вместо источников |

Число Jobs не обязано совпадать с числом действий в коде: `show()`, broadcast и материализация кэша могут порождать дополнительные внутренние Jobs. Объём Shuffle в строке repartition(4) мал, так как диагностический запрос использует только номер партиции, и остальные столбцы отсекаются оптимизатором.

## 16. Экспериментальные выводы по учебной части

1. **Ленивые вычисления.** Цепочка `select → filter → withColumn` над `geolocation` не создала ни одного Job; Job появился только после `count()` (0,556 с). Замер времени вокруг преобразований без действия измеряет лишь построение плана.
2. **Партиции и задачи.** Число входных партиций `geolocation` (16) совпало с числом задач первой стадии в Spark UI: одна задача обрабатывает одну партицию. При 20 вычислительных слотах четыре слота при чтении простаивали.
3. **Shuffle при groupBy.** Группировка `order_items` по `seller_id` включила `Exchange hashpartitioning(seller_id, 8)` между частичной и финальной агрегацией; Spark UI показал Shuffle Write 438,6 КиБ и стадию из 8 задач после обмена.
4. **repartition и coalesce.** При уменьшении перекошенного DataFrame с 16 до 4 партиций `repartition()` добавил `Exchange RoundRobinPartitioning(4)` и выровнял партиции (от 250 040 до 250 041 строки), а `coalesce()` использовал `Coalesce` без нового обмена и сохранил перекос (от 88 462 до 626 485 строк); в Spark UI запрос с `repartition` содержал на одну стадию больше.
5. **Sort Merge Join** перемешал и отсортировал обе стороны по `product_id` (Shuffle 3,74 МиБ), **Broadcast Hash Join** передал `products` через `BroadcastExchange` без перемешивания большой стороны (Shuffle 236 байт); число строк результата одинаково — 112 650. На стенде время составило 0,711 с против 0,448 с.
6. **cache/persist.** Повторные действия над сохранённым DataFrame заняли 0,384 и 0,196 с против 0,629 и 0,403 с без кэша; с учётом материализации (0,727 с) кэш при двух действиях не окупился (1,307 с против 1,032 с).
7. **Число партиций.** Минимальное медианное время бенчмарка получено при p = 1 при `defaultParallelism` = 20: при одной партиции Spark обходится без `Exchange`, а при p ≥ 2 объём перемешивания растёт пропорционально p. На объёме 2 млн строк накладные расходы перевешивают выигрыш от параллелизма.
8. **Маленькие файлы.** Число part-файлов совпало с числом партиций перед записью (1, 4, 16, 64), средний размер файла снизился с 6611,2 до 164,6 КиБ, а общий объём вырос с 6,46 до 10,29 МиБ за счёт метаданных и худшего сжатия мелких файлов.
9. **AQE.** С включённым AQE при 32 заданных партициях после перемешивания фактически выполнилась одна задача, что показывает изменение плана по runtime-статистике.

# 17. Самостоятельная работа. Вариант A: orders + customers

Основная таблица — `orders`, справочник — `customers`, ключ — `customer_id`, группировка — по `customer_state`. Справочник клиентов имеет тот же порядок размера, что и таблица заказов, а `order_status` обладает сильным перекосом (почти все заказы доставлены), что удобно для опыта с `repartition()`/`coalesce()`.

### 17.1. Данные

In [29]:
SELF_TABLES = ["orders", "customers"]

with sqlite3.connect(DB_PATH) as con:
    for table in SELF_TABLES:
        out = STAGING / f"{table}.csv"
        first = True
        for chunk in pd.read_sql_query(
            f"SELECT * FROM {table}", con, chunksize=100_000
        ):
            chunk.to_csv(
                out, index=False,
                mode="w" if first else "a",
                header=first, encoding="utf-8"
            )
            first = False

for name in SELF_TABLES:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(STAGING / f"{name}.csv"))
    )
    df.write.mode("overwrite").parquet(str(PARQUET / name))

orders = spark.read.parquet(str(PARQUET / "orders"))
customers = spark.read.parquet(str(PARQUET / "customers"))

print("orders:", orders.count(), "rows,", orders.rdd.getNumPartitions(), "partitions")
print("customers:", customers.count(), "rows,", customers.rdd.getNumPartitions(), "partitions")

orders: 99441 rows, 5 partitions
customers: 99441 rows, 3 partitions


Таблицы содержат по 99 441 строке; `customer_id` в справочнике уникален, поэтому соединение по нему не размножает заказы. Исходное число партиций: `orders` — 5, `customers` — 3.

### 17.2. Ленивые вычисления

In [30]:
sc.setJobGroup("S_LAZY", "Self: lazy evaluation")
delivered = (
    orders
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "delivery_days",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
    )
    .select("order_id", "customer_id", "delivery_days")
)
print(type(delivered).__name__)

DataFrame


In [31]:
t0 = perf_counter()
n = delivered.count()
print("Rows:", n)
print(f"Elapsed: {perf_counter() - t0:.3f} s")

Rows: 96478
Elapsed: 0.107 s


**Вывод.** После построения цепочки `filter → withColumn → select` в Spark UI не появилось Job группы `S_LAZY`; он возник только при `count()`, вернувшем 96 478 доставленных заказов.

### 17.3. repartition() и coalesce()

In [32]:
status16 = orders.repartition(16, "order_status")
s_repart4 = status16.repartition(4)
s_coal4 = status16.coalesce(4)

print("=== repartition(4) ===")
s_repart4.explain("formatted")
print("=== coalesce(4) ===")
s_coal4.explain("formatted")

print("repartition(4)")
partition_profile(s_repart4).show()
print("coalesce(4)")
partition_profile(s_coal4).show()

=== repartition(4) ===
== Physical Plan ==
Exchange (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#1805, customer_id#1806, order_status#1807, order_purchase_timestamp#1808, order_approved_at#1809, order_delivered_carrier_date#1810, order_delivered_customer_date#1811, order_estimated_delivery_date#1812]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/orders]
ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,order_approved_at:timestamp,order_delivered_carrier_date:timestamp,order_delivered_customer_date:timestamp,order_estimated_delivery_date:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#1805, customer_id#1806, order_status#1807, order_purchase_timestamp#1808, order_approved_at#1809, order_delivered_carrier_date#1810, order_delivered_customer_date#1811, order_estimated_delivery_date#1812]

| Вариант | min rows | max rows | mean rows |
|---|---|---|---|
| repartition(4) | 24 859 | 24 862 | 24 860 |
| coalesce(4) | 615 | 96 480 | 24 860 |

**Вывод.** Разбиение по `order_status` сильно перекошено, так как почти все заказы доставлены. `repartition(4)` с новым `Exchange RoundRobinPartitioning(4)` выровнял партиции, а `coalesce(4)` без нового обмена сохранил партицию с основной массой заказов.

### 17.4. groupBy по штату клиента

In [33]:
sc.setJobGroup("S_GROUPBY", "Self: groupBy customer_state")
state_stats = (
    customers
    .groupBy("customer_state")
    .agg(F.count("*").alias("customers"))
    .orderBy(F.desc("customers"))
)
state_stats.show(5, truncate=False)
state_stats.explain("formatted")

+--------------+---------+
|customer_state|customers|
+--------------+---------+
|SP            |41746    |
|RJ            |12852    |
|MG            |11635    |
|RS            |5466     |
|PR            |5045     |
+--------------+---------+
only showing top 5 rows
== Physical Plan ==
* Sort (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- Exchange (4)
         +- * HashAggregate (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_state#1817]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/customers]
ReadSchema: struct<customer_state:string>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [customer_state#1817]

(3) HashAggregate [codegen id : 1]
Input [1]: [customer_state#1817]
Keys [1]: [customer_state#1817]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#1884L]
Results [2]: [customer_state#1817, count#1885L]

(4) Exchang

**Наблюдение в Spark UI.** Stages: 2; Tasks по стадиям: 3 → 8; Shuffle Write: 2,4 КиБ; Shuffle Read: 2,4 КиБ.

**Вывод.** Группировка по `customer_state` содержит `Exchange hashpartitioning(customer_state, 8)` между частичной и финальной агрегацией. Объём перемешивания мал, потому что частичная агрегация сводит каждую входную партицию к 27 группам ещё до обмена.

### 17.5. Sort Merge Join и Broadcast Hash Join

In [34]:
sc.setJobGroup("S_SMJ", "Self: SMJ orders x customers")
s_smj = (
    orders.hint("merge")
    .join(customers.hint("merge"), on="customer_id", how="inner")
)
s_smj.explain("formatted")

t0 = perf_counter()
print("Rows:", s_smj.count())
print(f"Elapsed: {perf_counter() - t0:.3f} s")

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [8]: [order_id#1805, customer_id#1806, order_status#1807, order_purchase_timestamp#1808, order_approved_at#1809, order_delivered_carrier_date#1810, order_delivered_customer_date#1811, order_estimated_delivery_date#1812]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/orders]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,order_approved_at:timestamp,order_delivered_carrier_date:timestamp,order_delivered_customer_date:timestamp,order_estimated_delivery_date

In [35]:
sc.setJobGroup("S_BHJ", "Self: BHJ orders x customers")
s_bhj = orders.join(
    F.broadcast(customers),
    on="customer_id",
    how="inner"
)
s_bhj.explain("formatted")

t0 = perf_counter()
print("Rows:", s_bhj.count())
print(f"Elapsed: {perf_counter() - t0:.3f} s")

== Physical Plan ==
* Project (9)
+- * BroadcastHashJoin Inner BuildRight (8)
   :- * Filter (3)
   :  +- * ColumnarToRow (2)
   :     +- Scan parquet  (1)
   +- BroadcastExchange (7)
      +- * Filter (6)
         +- * ColumnarToRow (5)
            +- Scan parquet  (4)


(1) Scan parquet 
Output [8]: [order_id#1805, customer_id#1806, order_status#1807, order_purchase_timestamp#1808, order_approved_at#1809, order_delivered_carrier_date#1810, order_delivered_customer_date#1811, order_estimated_delivery_date#1812]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/krezon/Education/Sem7/Big Data/Tasks/Tasks3-4/spark_input_parquet/orders]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,order_approved_at:timestamp,order_delivered_carrier_date:timestamp,order_delivered_customer_date:timestamp,order_estimated_delivery_date:timestamp>

(2) ColumnarToRow [codegen id : 2]
Input [8]: [order

| Показатель | Sort Merge Join | Broadcast Hash Join |
|---|---|---|
| Строк результата | 99 441 | 99 441 |
| Exchange большой стороны | есть: `hashpartitioning(customer_id, 8)` | нет |
| Shuffle Read / Write | 6,42 МиБ / 6,42 МиБ | 295 Б / 295 Б |
| Jobs / Stages | 1 / 4 | 2 / 3 |
| Время, с | 0,392 | 0,233 |

**Вывод.** Обе стратегии вернули 99 441 строку. Sort Merge Join перемешал обе таблицы по `customer_id` (по 3,21 МиБ с каждой стороны), Broadcast Hash Join передал `customers` всем задачам отдельным Job и не перемешивал `orders`. На стенде время составило 0,392 с против 0,233 с. В отличие от учебного опыта, здесь broadcast-сторона почти так же велика, как основная таблица, поэтому выигрыш broadcast сильнее зависит от памяти исполнителей.

### 17.6. Эффект persist(MEMORY_AND_DISK)

In [36]:
spark.catalog.clearCache()

s_joined = delivered.join(
    F.broadcast(customers.select("customer_id", "customer_state", "customer_city")),
    "customer_id", "left"
)


def s_action_state(df):
    return (
        df.groupBy("customer_state")
        .agg(F.avg("delivery_days").alias("avg_days"))
        .agg(F.sum("avg_days").alias("checksum"))
        .first()[0]
    )


def s_action_city(df):
    return (
        df.groupBy("customer_city")
        .count()
        .agg(F.sum("count").alias("checksum"))
        .first()[0]
    )


sc.setJobGroup("S_NOCACHE", "Self: without persist")
t0 = perf_counter()
_ = s_action_state(s_joined)
t1 = perf_counter()
_ = s_action_city(s_joined)
t2 = perf_counter()
print(f"Action 1: {t1 - t0:.3f} s")
print(f"Action 2: {t2 - t1:.3f} s")

Action 1: 0.407 s
Action 2: 0.406 s


In [37]:
sc.setJobGroup("S_CACHE", "Self: persist")
s_cached = s_joined.persist(StorageLevel.MEMORY_AND_DISK)

t0 = perf_counter()
s_cached.count()
t1 = perf_counter()
_ = s_action_state(s_cached)
t2 = perf_counter()
_ = s_action_city(s_cached)
t3 = perf_counter()

print(f"Materialization: {t1 - t0:.3f} s")
print(f"Cached action 1: {t2 - t1:.3f} s")
print(f"Cached action 2: {t3 - t2:.3f} s")

s_cached.unpersist(blocking=True)
print("После unpersist:", s_cached.is_cached)

Materialization: 0.420 s
Cached action 1: 0.259 s
Cached action 2: 0.283 s
После unpersist: False


| Вариант | Материализация, с | Action 1, с | Action 2, с |
|---|---|---|---|
| Без persist | — | 0,407 | 0,406 |
| MEMORY_AND_DISK | 0,420 | 0,259 | 0,283 |

**Вывод.** Промежуточный DataFrame (доставленные заказы со штатом и городом клиента) использован двумя действиями. Повторные действия над кэшем заняли 0,259 и 0,283 с против 0,407 и 0,406 с без него; с учётом материализации (0,420 с) суммарное время составило 0,962 с против 0,813 с, то есть кэш не окупился.

### 17.7. Серия по числу партиций

Таблица доставленных заказов разбивается на p партиций и сохраняется в кэше, чтобы измерялась только агрегация над p партициями, а не само перераспределение. Для каждого p — прогрев и три замера, итог — медиана.

In [38]:
S_P_VALUES = [1, 2, 4, 8, 16]


def run_series(p, repeats=3):
    df_p = delivered.repartition(p).persist(StorageLevel.MEMORY_AND_DISK)
    df_p.count()  # материализация, не входит в замер

    def action():
        return df_p.groupBy("customer_id").agg(F.max("delivery_days")).count()

    action()  # прогрев
    times = []
    for _ in range(repeats):
        t0 = perf_counter()
        action()
        times.append(perf_counter() - t0)
    df_p.unpersist(blocking=True)
    return times, median(times)


s_results = []
for p in S_P_VALUES:
    times, med = run_series(p)
    s_results.append((p, *times, med))

t_1 = s_results[0][-1]
for p, *runs, med in s_results:
    print(f"p={p:2d}  runs={[round(t, 3) for t in runs]}  median={med:.3f} s  S(p)={t_1 / med:.2f}")

p= 1  runs=[0.123, 0.114, 0.099]  median=0.114 s  S(p)=1.00
p= 2  runs=[0.201, 0.185, 0.21]  median=0.201 s  S(p)=0.57
p= 4  runs=[0.141, 0.168, 0.146]  median=0.146 s  S(p)=0.78
p= 8  runs=[0.187, 0.186, 0.174]  median=0.186 s  S(p)=0.61
p=16  runs=[0.249, 0.256, 0.243]  median=0.249 s  S(p)=0.46


| p | Run 1, с | Run 2, с | Run 3, с | Median, с | S(p) |
|---|---|---|---|---|---|
| 1 | 0,123 | 0,114 | 0,099 | 0,114 | 1,00 |
| 2 | 0,201 | 0,185 | 0,210 | 0,201 | 0,57 |
| 4 | 0,141 | 0,168 | 0,146 | 0,146 | 0,78 |
| 8 | 0,187 | 0,186 | 0,174 | 0,186 | 0,61 |
| 16 | 0,249 | 0,256 | 0,243 | 0,249 | 0,46 |

**Вывод.** Минимальное медианное время 0,114 с получено при p = 1 (S = 1,00). Как и в учебном бенчмарке, при одной партиции группировка выполняется одной стадией без `Exchange`, а при p ≥ 2 каждый запуск перемешивает около 3,1 МиБ частичных результатов (почти каждый `customer_id` встречается один раз, и частичная агрегация данные не сокращает). Среди вариантов с перемешиванием лучшим оказалось p = 4 (0,146 с); при p = 16 время выросло до 0,249 с из-за накладных расходов на задачи.

### 17.8. Запись итогового DataFrame с разным числом партиций

In [39]:
OUT_SELF = Path("self_files_experiment")
OUT_SELF.mkdir(exist_ok=True)

s_final = s_joined.select("order_id", "customer_state", "customer_city", "delivery_days")

for p in [1, 8, 32]:
    path = OUT_SELF / f"p_{p}"
    s_final.repartition(p).write.mode("overwrite").parquet(str(path))
    files = list(path.glob("part-*.parquet"))
    sizes = [f.stat().st_size for f in files]
    print(p, len(files), round(sum(sizes) / 1024**2, 2), round(sum(sizes) / len(sizes) / 1024, 1))

1 1 3.32 3402.3
8 8 3.41 436.5
32 32 3.56 113.9


| p | part-файлов | Общий размер, MiB | Средний файл, KiB |
|---|---|---|---|
| 1 | 1 | 3,32 | 3402,3 |
| 8 | 8 | 3,41 | 436,5 |
| 32 | 32 | 3,56 | 113,9 |

**Вывод.** Число part-файлов равно числу партиций перед записью; средний размер файла уменьшается с ростом p (с 3402,3 до 113,9 КиБ), а общий объём немного растёт (с 3,32 до 3,56 МиБ) за счёт метаданных каждого файла.

### 17.9. Выводы и ограничения

**Экспериментальные выводы**

1. Цепочка из трёх преобразований над `orders` не создала Job; вычисление запустилось только при `count()`, вернувшем 96 478 доставленных заказов.
2. При перекошенном разбиении по `order_status` `repartition(4)` выровнял партиции (от 24 859 до 24 862 строк) ценой нового `Exchange`, а `coalesce(4)` сохранил перекос (от 615 до 96 480 строк).
3. Группировка по `customer_state` содержит `Exchange hashpartitioning`; Shuffle Write составил всего 2,4 КиБ благодаря частичной агрегации до обмена.
4. JOIN `orders` и `customers` дал одинаковое число строк в обеих стратегиях (99 441); Sort Merge Join перемешал обе стороны (Shuffle Write 6,42 МиБ), Broadcast Hash Join — ни одной (295 байт на финальный подсчёт); время на стенде 0,392 с против 0,233 с.
5. `persist()` при двух повторных действиях не окупился: 0,962 с с учётом материализации против 0,813 с без кэша, хотя каждое повторное действие ускорилось.
6. В серии по числу партиций лучшее время получено при p = 1, когда группировка обходится без перемешивания; число part-файлов при записи совпало с p (1, 8, 32).

**Ограничения измерений**

1. Опыты выполнены в режиме `local[*]` на одном компьютере: перемешивание идёт между потоками, а не по сети, поэтому абсолютные времена и выигрыш стратегий не переносятся на кластер.
2. Времена измерены сериями по одному–три запуска на одном стенде и зависят от прогрева JVM, файлового кэша ОС и фоновой нагрузки; при объёме около 100 тыс. строк операции длятся доли секунды, и значительную долю времени занимают накладные расходы на планирование задач.

## 18.1. Завершение сессии

In [40]:
spark.stop()